# Mini-Series: Memory Without Words
>> **How Neural Networks Represent and Remember Knowledge**

# INTRODUCTION: Symbolic vs. Latent

Information can be represented in variety of forms. For the purposes of this mini-series there are two important categories:

- **Symbolic**

    - text in any of written languages
    - voice in any of spoken languages
    - gestures
    - images
    - videos

- **Latent**

    - internal language of a specific processing machine (LLM, DB, ...)
    - internal activations inside a neural brain (human or AI)

- Symbolic form is good for **persistent cross-boundary** storage:

    - cross-epoch in human history
    - cross-culture in human civilization
    - cross-processors
    - cross-episodes of LLM inference

- Latent form is optimized for using **inside** a specific processor:

    - shaped for specific API
    - packed in specific representation (dimensionality, accuracy)
    - transoformed into specific geometry perspective

**Descending from high philosophical level to the gound of LLMs:**

Where the information lives in LLM?

<table>
    <tr>
        <td><b>Outside: Symbols / Words</b></td><td><b>Inside: Latent / Numbers</b></td>
    </tr>
    <tr>
        <td><img src="P2_MemoryInWords.jpg"></td>
        <td><img src="P1_MemoryInBrain.jpg"></td>
    </tr>
</table>


- **Symbolic**
    - RAG-storages indexed by context similarity
    - WEB pages found by search requests
    - Corporate sources - emails, DB, ...
- **Latent**
    - pretrained weights - huge constant tensors
    - external latent memory banks (NTM, DNC, Memorizing Transformers, ...)

### **Why this distinction is important**

- **Different lifetimes:**  
  Symbolic information persists across time and systems; latent information is ephemeral, living only within the active state of a model.

- **Different accessibility:**  
  Symbolic data can be read, indexed, and shared; latent data is private to the processor — meaningful only inside its representational geometry.

- **Different operations:**  
  Symbolic form supports search, retrieval, and reasoning through explicit structure; latent form supports fast, implicit reasoning through learned geometry.

- **Different bridges:**  
  To connect external memory (RAG, databases, human notes) with internal reasoning (LLM activations, latent memory modules), we must understand how to **translate between symbolic and latent spaces**.

- **Different failure modes:**  
  Symbolic systems fail by omission or ambiguity; latent systems fail by distortion — when geometry no longer preserves meaning.

Understanding this boundary between symbolic and latent representation is the key to designing systems that can **store, recall, and evolve knowledge** — not just retrieve words.

# PART A. Knowledge Representation and Geometries

We will be talking about knowledge representation, accumulation, transformation, and recall in LLMs. So the natural starting question is:

**How is knowledge represented inside a transformer?**

From previous lectures, you already know part of the answer:

> Knowledge lives in the *representations* — the collection of **d_model‑dimensional vectors** produced at every layer, for every token position.

- token position in every layer contains a representation of a **single word**
- the whole layer is a representation of the **whole input sequence**
- earliest layer (top) - **input tokens** embeddings
- latest layer (bottom) - **output tokens** embeddings

<br/>
<img src="IMG-5-01-3D transformer layer.png" />

<details>
<summary>Details: <strong>Representation Enigma</strong></summary>

- A transformer layer maps a sequence of tokens into a **context_length × d_model** matrix.

- Across all layers, the model builds a *stack* of such matrices — each one a different transformation of the same input.

- *(Typical dimensions: Layers ≈ 80–120, d_model ≈ 8K-12K, context_length ≈ 8K–32K)*

You’ve also heard the informal idea that different layers correspond to **different conceptual perspectives**:  
grammar → syntax → entities → relations → world facts → reasoning.

That phrasing is hand‑wavy.  
Today we will replace it with something more concrete, mathematical, and inspectable.

### The Representation Enigma

All layers of a transformer operate in the **same ℝᵈ vector space**.  
Every hidden state (at an input token position) is “just” a point in ℝᵈ.

This raises deep questions:

- **What does a point in this space *mean*?**  
- **How does meaning change as we move from layer to layer?**  
- **Are points from different layers comparable?**  
- **What does it mean for a layer to represent a specific “conceptual perspective” if the space is the same?**

To build intuition, we will start with a deliberately simple system:  
a small multi‑layer neural network trained on the classic XOR dataset.

This is not a transformer — and that’s the point.  
By stripping away architectural complexity, we can focus on the **geometry** of representation.

### What this part will show

- All layers live in the **same ℝᵈ space**  
- Yet the **geometry of the data** changes across layers  
- Non‑separable patterns become **linearly separable**  
- Representations become **structured**, enabling later computation  
- This gives us a concrete model of **latent memory as intermediate geometry**
- **Each layer constructs its own internal “language” — a basis of directions and geometric structures that encode meaning. Understanding these layer‑specific languages is essential if we want to extract, store, recall, or compare pieces of latent knowledge.**

This will set the stage for understanding how transformers accumulate and transform knowledge — not symbolically, but **geometrically**.

</details>


# 🧠 SECTION 0 — Imports & Setup

In [ ]:
# Cell 0: Imports

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# 🧠 SECTION 1 — Nonlinear Task (XOR)

We construct a task that is not linearly separable in input space.

This is crucial because it forces the network to build a **new representation space** - giving us a chance to compare the spaces.

In [ ]:
# Cell 1: Create XOR-like dataset
# A 2D XOR dataset — the classic example of something not linearly separable. 

# Create 400 random 2D points
N = 400
X = torch.randn(N, 2)

# Nonlinear function: XOR of signs (in ML terms: y[i] - a label for respective point in X[i])
y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).float().unsqueeze(1)
# Breakdown of the above expression:
#     - X[:, 0] > 0 → True if the point is on the right half-plane
#     - X[:, 1] > 0 → True if the point is in the upper half-plane
#     - ^ is XOR (exclusive OR)

# Now take a look into the X representation space as a 2D geometry
plt.figure()
plt.scatter(X[:, 0], X[:, 1], c=y.squeeze(), cmap='bwr') # 0=>"b" 1=>"r"
plt.title("Input Space (Not Linearly Separable)")
plt.show()

<details>
<summary>Details: <strong>What do axes represent in this representation space?</strong></summary>

Answer: **Nothing clearly meaningful!**

Each of the two dimensions represent a mixture of meanings which could be expressed by a mathematical expression, whose meaning is not easy to use:

The point (x,y) means 1 (blue) if:
  - x > 0 and y < 0  (Quadrant IV)
  - x < 0 and y > 0  (Quadrant II)
  
And 0 (red) if:
  - x > 0 and y > 0  (Quadrant I)
  - x < 0 and y < 0  (Quadrant III)

To be able to more easily reason about the points we need a different perspective (geometry) for the same points in which each of the dimensions would be easier to read.

By stacking several neural layers we can try to find such more convenient geometries.
</details>

<details>
<summary>Details: <strong>Why this dataset matters</strong></summary>

This is the simplest example showing that:

- A linear model fails in principle for such dataset
- A neural network with a hidden layer succeeds
- Because the decision boundary must be nonlinear (curved or piecewise)

It’s the canonical demonstration of why deep learning is useful. **We'll see why it matters in LLM.**

</details>


# 🧠 SECTION 2 — Define Small Network

We keep dimension constant across layers to mirror transformers.

Let’s use width = 2 so we can visualize directly.

In [ ]:
# Cell 2: Small MLP with constant hidden dimension

class SmallMLP(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.l1 = nn.Linear(2, d)
        self.l2 = nn.Linear(d, d)
        self.l3 = nn.Linear(d, 1)
        self.act = nn.Tanh()
        
    def forward(self, x):
        h1 = self.act(self.l1(x))
        h2 = self.act(self.l2(h1))
        out = self.l3(h2)
        return out, h1, h2  # returning the prediction AND internal layer representations so we can visualize

model = SmallMLP(d=2)

# 🧠 SECTION 3 — Train

In [ ]:
# Cell 3a: Train model

# Optimizer: 
# Adam is an adaptive gradient‑based optimizer that adjusts the step size for each parameter individually.
# Combines the benefits of momentum (smooths updates) and RMSProp (scales updates by recent gradient magnitudes).
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Loss function: 
# nn.BCEWithLogitsLoss() is the standard loss function for binary classification in PyTorch — 
# but with an important twist: it expects raw logits, not probabilities.
loss_fn = nn.BCEWithLogitsLoss()

# Training loop:
for epoch in range(2000):
    optimizer.zero_grad()
    logits, _, _ = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

# 
print("Final loss:", loss.item())

<details>
<summary>Details: <strong>How to read the number intuitively</strong></summary>

**Binary cross‑entropy for a single example is**:
```
0.5 = log (probability(correct))
```

So you can invert it:
```
probability(correct) ~~ exp(-0.5) ~~ 0.61 (61%)
```

**This means:**
- The model is not guessing (0.5 loss would be random guessing with perfect calibration)
- The model is not confident (good models get BCE ≪ 0.1)
- The model is giving the correct class about 60% probability on average

</details>

# 🧠 SECTION 4 — Visualize Representation Spaces

In [ ]:
# Cell 4a: Extract layer representations

with torch.no_grad():
    _, h1, h2 = model(X)

h1 = h1.numpy()
h2 = h2.numpy()
X_np = X.numpy()
y_np = y.numpy().squeeze()

In [ ]:
# Cell 4b: Visualize geometry evolution

# Plot all spaces side by side

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_np[:,0], X_np[:,1], c=y_np, cmap='bwr')
axes[0].set_title("Input Space")

axes[1].scatter(h1[:,0], h1[:,1], c=y_np, cmap='bwr')
axes[1].set_title("Layer 1 Representation")

axes[2].scatter(h2[:,0], h2[:,1], c=y_np, cmap='bwr')
axes[2].set_title("Layer 2 Representation")

plt.show()

# 🧠 SECTION 5 — Probe Linear Separability

**Can a straight line separate the classes?**

The scatter plots above show the geometry, but leave a question open: *how* linearly separable is each layer, exactly?

We train a single linear classifier on each representation and draw its decision boundary directly onto the point cloud. If the boundary cleanly divides red from blue, the layer has done its job. If not, no straight line in that space can help — no matter how you draw it.

In [ ]:
# Cell 5a: Decision boundary overlay on each representation space

def fit_linear_boundary(features: np.ndarray, labels: np.ndarray):
    """Train a logistic linear probe and return the probe model."""
    probe = nn.Linear(2, 1)
    opt = optim.Adam(probe.parameters(), lr=0.05)
    loss_fn = nn.BCEWithLogitsLoss()

    Xp = torch.tensor(features, dtype=torch.float32)
    yp = torch.tensor(labels.reshape(-1, 1), dtype=torch.float32)

    for _ in range(500):
        opt.zero_grad()
        loss_fn(probe(Xp), yp).backward()
        opt.step()
    return probe

def draw_boundary(ax, probe, xlim: tuple, ylim: tuple, n: int = 300) -> None:
    """Draw the linear decision boundary and shaded regions."""
    # build a dense grid over the plot area
    xs: np.ndarray = np.linspace(xlim[0], xlim[1], n)
    ys: np.ndarray = np.linspace(ylim[0], ylim[1], n)
    xx, yy = np.meshgrid(xs, ys)
    grid = torch.tensor(
        np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32
    )
    with torch.no_grad():
        zz: np.ndarray = torch.sigmoid(probe(grid)).numpy().reshape(xx.shape)
    # soft shading of class regions
    ax.contourf(xx, yy, zz, levels=50, cmap="bwr", alpha=0.25, vmin=0, vmax=1)
    # hard decision boundary at p=0.5
    ax.contour(xx, yy, zz, levels=[0.5], colors="k", linewidths=2)


datasets: list[tuple[str, np.ndarray]] = [
    ("Input Space", X_np),
    ("Layer 1 Representation", h1),
    ("Layer 2 Representation", h2),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (title, feats) in zip(axes, datasets):
    # fit a fresh linear probe for this representation
    probe = fit_linear_boundary(feats, y_np)
    with torch.no_grad():
        preds: torch.Tensor = torch.sigmoid(probe(torch.tensor(feats, dtype=torch.float32))) > 0.5
        acc: float = (preds.float() == torch.tensor(y_np.reshape(-1,1), dtype=torch.float32)).float().mean().item()

    # compute plot limits with a small margin
    margin: float = 0.3
    xlim: tuple = (feats[:, 0].min() - margin, feats[:, 0].max() + margin)
    ylim: tuple = (feats[:, 1].min() - margin, feats[:, 1].max() + margin)

    draw_boundary(ax, probe, xlim, ylim)
    ax.scatter(feats[:, 0], feats[:, 1], c=y_np, cmap="bwr",
               edgecolors="#333", linewidths=0.3, s=25, zorder=3)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(f"{title}\nlinear probe accuracy: {acc:.0%}", fontsize=10)
    ax.set_xlabel("dim 0")
    ax.set_ylabel("dim 1")

fig.suptitle(
    "Same points, same labels — only the coordinate system changes.\n"
    "The black line is the best possible straight-line boundary. It fails in input space; it works in Layer 2.",
    fontsize=10, y=1.03
)
plt.tight_layout()
plt.show()


### Bingo

**Initially messy dimensions (left diagram) became almost othrogonolazed: the point have two (almost) independent features - grades of "blueness" and "redness".**

The black line is identical in kind across all three panels — one weight, one bias, one straight cut through 2D space. What differs is the space it is cutting through.

In input space the line cannot succeed: no matter where you draw it, XOR points will straddle it. In Layer 2 space the same kind of line almost perfectly separates the classes — because the network has **learned a coordinate system in which the answer is readable by a ruler**.

This is the key insight before discussing latent representations: when the network writes a vector into a memory slot, it is writing a coordinate in *this* kind of space — not raw input, not text, but in a geometry deliberately shaped so that useful distinctions lie along straight directions. That is what makes cosine-similarity addressing meaningful rather than arbitrary.


### Now pause:

- All points are in ℝ².
- But the geometry changed.
- Layer 2 makes classes nearly linearly separable.

This is the “different representation spaces” moment:

- Deep networks don't change dimensionality.
- They change the **geometric perspective** to the same set of objects.

# 🧠 SECTION 6 — Same Vector, Different Meaning

Now the philosophical point.

Pick a single vector from layer 2:

In [ ]:
# Cell 6a: Same vector, different linear decoders

v = torch.tensor(h2[0], dtype=torch.float32)

# Two random decoders
decoder_A = torch.randn(2)
decoder_B = torch.randn(2)

print("Decoder A output:", torch.dot(decoder_A, v).item())
print("Decoder B output:", torch.dot(decoder_B, v).item())

Explain:

- The vector has no intrinsic meaning.
- Meaning emerges from the linear map applied to it.

This sets up latent memory discussion beautifully.

# 🧠 SECTION 7 — The Takeaway


**What is a Representation Space?**

All layers live in the same vector space ℝᵈ.

What changes across layers is:

- The geometry
- Which features are linearly separable
- Which directions correspond to useful abstractions

A representation space is:

>> A coordinate system in which the remaining computation becomes easier.

Deep learning = progressive linearization of useful features.

**Why this works as introduction to latent memory**

- Latent Memory is not just stored tokens (as opposite to text storage)
- It is structured latent state - aligned with particular representation space
- Each layer reshapes the state to expose different abstractions
- Latent memory architectures manipulate these representation spaces